In [0]:
%sql
USE CATALOG maven_market_uc;

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS checkpoint_location
URL 'abfss://checkpoint@sgmavenmarket.dfs.core.windows.net/'
WITH (CREDENTIAL `maven-storage-cred`)
COMMENT 'Direct access to checkpoint container';

In [0]:
from pyspark.sql.functions import current_timestamp, lit

kafka_server = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
api_key      = "OLTWWZBAC3VVR7SA"
secret       = "cfltD7VBSW9eBXMcqED6k+XPnOvRLauV94foRqq3bzYwobjIBTjtItwKV5biUBYQ"

sasl_config = (
    f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
    f"required username='{api_key}' password='{secret}';"
)

BASE_CHECKPOINT = "abfss://checkpoint@sgmavenmarket.dfs.core.windows.net/"

def read_kafka_topic(topic):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", kafka_server)
        .option("subscribe", topic)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl_config)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")   # safety net
        .load()
        .selectExpr(
            "CAST(value AS STRING) as raw_payload",
            "topic as _kafka_topic",
            "partition as _kafka_partition",
            "offset as _kafka_offset",
            "timestamp as _kafka_timestamp"
        )
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("kafka"))
    )

In [0]:
df_inventory = read_kafka_topic("inventory")

query_inventory = (
    df_inventory.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BASE_CHECKPOINT + "inventory/")  # ← unique path
    .toTable("maven_market_uc.bronze.kafka_inventory")
)

In [0]:
%sql
SELECT * FROM maven_market_uc.bronze.kafka_inventory
ORDER BY _ingestion_timestamp DESC
LIMIT 20;

In [0]:
# ============================================================
# CELL 5 - Orders stream
# ============================================================
df_orders = read_kafka_topic("orders")

query_orders = (
    df_orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BASE_CHECKPOINT + "orders/")    # ← unique path
    .toTable("maven_market_uc.bronze.kafka_orders")
)

In [0]:
%sql
SELECT * FROM maven_market_uc.bronze.kafka_orders
ORDER BY _ingestion_timestamp DESC
LIMIT 20;

In [0]:
# ============================================================
# CELL 6 - Monitor both streams (optional)
# ============================================================
import time

for _ in range(5):
    print("=== Inventory ===", query_inventory.status)
    print("=== Orders ===",    query_orders.status)
    time.sleep(10)

In [0]:
%sql
SELECT COUNT(*) as total_rows, MAX(_ingestion_timestamp) as last_seen
FROM maven_market_uc.bronze.kafka_inventory;

In [0]:
%sql
SELECT COUNT(*) as total_rows, MAX(_ingestion_timestamp) as last_seen
FROM maven_market_uc.bronze.kafka_orders;